# Heart Rate Forecasting - TCN (Temporal Convolutional Network)

Modelo creado por el profesor. Usa TensorFlow/Keras con la librería keras-tcn.

In [ ]:
# Setup: clonar repo e instalar dependencias
!git clone https://github.com/AllanDBB/heartrate-forecasting.git
%cd heartrate-forecasting
!pip install -q -r requirements.txt
!pip install -q keras-tcn tensorflow "numpy<2.0"
import os; os.kill(os.getpid(), 9)  # Reinicia el kernel automáticamente

In [ ]:
# Verificar GPU
import tensorflow as tf
print("Num GPUs:", len(tf.config.list_physical_devices('GPU')))
if tf.config.list_physical_devices('GPU'):
    print("Device:", tf.config.list_physical_devices('GPU')[0])

In [ ]:
# Cargar y preparar datos
%cd /content/heartrate-forecasting

import utils
from sklearn.model_selection import train_test_split

folder_path = 'dataset/'
dfSelectedMean = utils.loadAllFiles(folder_path)

inp = 200
out = 200

df_70, df_30 = utils.selectRandomColumns(dfSelectedMean)

pathEst_70 = 'dataset/values_deses_70.csv'
pathEst_30 = 'dataset/values_deses_30.csv'

df_scaled_70, values_deses_70 = utils.estandarizar(df_70, pathEst_70)
df_scaled_30, values_deses_30 = utils.estandarizar(df_30, pathEst_30)

X_train, y_train, ids_list_tr = utils.series_to_supervised_matrix(
    df_scaled_70.iloc[:, :-2], input_size=inp, output_size=out
)
X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

X_test, y_test, ids_list_te = utils.series_to_supervised_matrix(
    df_scaled_30.iloc[:, :-2], input_size=inp, output_size=out
)

print('Datasets splitted and ready!')
print('Shapes:', X_tr.shape, y_tr.shape, X_va.shape, y_va.shape, X_test.shape, y_test.shape)

## Opción A: Cargar modelo pre-entrenado del profesor

In [ ]:
import os, sys
os.chdir('/content/heartrate-forecasting')
sys.path.insert(0, '/content/heartrate-forecasting')

from wrappers.TCNSupervisedWrapper import TCNSupervisedWrapper

# Cargar modelo pre-entrenado (tcn_0.keras)
wrapper = TCNSupervisedWrapper(config_path='configs/tcn_config.yaml')
print("Modelo cargado:", wrapper.model is not None)

## Opción B: Entrenar desde cero (fine-tuning)
Si no existe un modelo pre-entrenado o queremos re-entrenar:

In [ ]:
# Descomentar para entrenar desde cero:
# wrapper.model = None  # Forzar construcción desde cero
# history, fig_loss = wrapper.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)

## Evaluación con métricas avanzadas
Incluye MAE, RMSE, MAPE, DTW, DDTW y Cross-Correlation.

In [ ]:
# Evaluación sobre test set con todas las métricas
metrics_test = wrapper.evaluate(X_test, y_test)

In [ ]:
# Evaluación sobre validation set
metrics_val = wrapper.evaluate(X_va, y_va)

## Visualización de Resultados
Gráficos intuitivos para entender el rendimiento del modelo.

In [ ]:
# Obtener predicciones para visualización
y_pred_test = wrapper.predict(X_test)

# 1. Ejemplos de predicción vs realidad
utils.plot_forecast_samples(y_test, y_pred_test, n_samples=6,
                            title="TCN: Predicción vs Real (muestras)")

In [ ]:
# 2. Error según horizonte de predicción
utils.plot_error_over_horizon(y_test, y_pred_test,
                              title="TCN: Error según horizonte de predicción")

In [ ]:
# 3. Rendimiento por sujeto
utils.plot_subject_performance(y_test, y_pred_test, ids_list_te,
                               title="TCN: Rendimiento por Sujeto")

In [ ]:
# 4. Línea de tiempo completa para un sujeto específico
utils.plot_prediction_timeline(y_test, y_pred_test, ids_list_te, inp,
                               subject_idx=0,
                               title="TCN: Línea de Tiempo Completa")